# NB03 — Apple FullBody Reward Ground Truth, Safety & MLflow

This notebook is the **single source of truth** for reward/safety analysis of `UnitreeG1PlaceAppleInBowlFullBody-v1`.

Key upgrades vs older NB03:
- Reward Contract is auto-aligned to `apple_fullbody_env.py` (weights, success, NaN failure)
- Reward **decomposition + sanity checks** (dense vs normalized_dense)
- **Pretty visualizations** (episode filmstrip + time-series plots)
- **Error analysis** (NaN/explosion vs timeout, action/jerk statistics, outlier detection)
- MLflow logging of metrics + artifacts


## Objectives

1. Import + register the custom env from `apple_fullbody_env.py` (ground-truth).
2. Generate a **ground-truth Reward Contract** (JSON) that matches the env *exactly*.
3. Run evaluation rollouts (random + heuristic) and:
   - validate `info` keys (`success`, `fail`, distance metrics)
   - compute **reward component decomposition** and confirm totals
4. Produce visualizations:
   - reward/action/jerk distributions
   - termination breakdown (`success`, `fail_nan`, `timeout`)
   - episode filmstrip (rendered frames + overlays)
5. Produce an error-analysis report and log everything to MLflow.


## Resources

- Env ground-truth: `apple_fullbody_env.py`
- Env ID: `UnitreeG1PlaceAppleInBowlFullBody-v1`
- Control: `pd_joint_delta_pos` (37 DOF)
- Reward modes: `dense`, `normalized_dense`, `sparse`, `none`

> Note: this notebook intentionally treats **NaN / simulation explosion** as the primary failure mode (per env `evaluate()`), not "fall detection".


## Imports & Setup


In [ ]:
import sys, os, json, random
from pathlib import Path

# (Optional) Force Lavapipe software Vulkan (useful on headless GPU nodes)
os.environ.setdefault("VK_ICD_FILENAMES", "/usr/share/vulkan/icd.d/lvp_icd.json")
os.environ.setdefault("MESA_VK_DEVICE_SELECT", "10005:0")

import numpy as np
import torch
import gymnasium as gym
import matplotlib.pyplot as plt

import mani_skill.envs
from mani_skill.utils.wrappers.gymnasium import CPUGymWrapper

# ---------------------------------------------------------------------
# Robust project root detection (works in VSCode, Colab, RunPod)
# ---------------------------------------------------------------------
try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent
except NameError:
    _cwd = Path.cwd()
    if (_cwd / "src" / "envs").is_dir():
        PROJECT_ROOT = _cwd
    elif (_cwd.parent / "src" / "envs").is_dir():
        PROJECT_ROOT = _cwd.parent
    else:
        PROJECT_ROOT = _cwd  # fallback

sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------------------
# Import the env module (ground truth). We support both layouts:
#   - repo-root: apple_fullbody_env.py
#   - src/envs/: apple_fullbody_env.py or re-export via src.envs
# ---------------------------------------------------------------------
ENV_MODULE_PATHS = [
    PROJECT_ROOT / "apple_fullbody_env.py",
    PROJECT_ROOT / "src" / "envs" / "apple_fullbody_env.py",
    Path.cwd() / "apple_fullbody_env.py",
]

def _dynamic_import_from_path(py_path: Path, module_name: str):
    import importlib.util
    spec = importlib.util.spec_from_file_location(module_name, str(py_path))
    mod = importlib.util.module_from_spec(spec)
    assert spec and spec.loader
    spec.loader.exec_module(mod)  # registers env via decorators
    return mod

env_mod = None
env_mod_name = None

# 1) try standard imports
try:
    import src.envs as _src_envs
    env_mod = _src_envs
    env_mod_name = "src.envs"
except Exception:
    pass

# 2) try dynamic import
if env_mod is None:
    for p in ENV_MODULE_PATHS:
        if p.exists():
            env_mod = _dynamic_import_from_path(p, "apple_fullbody_env")
            env_mod_name = str(p)
            break

assert env_mod is not None, "❌ Cannot import apple_fullbody env module. Check paths."

print(f"✅ Loaded env module from: {env_mod_name}")

# Artifacts dir
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "NB03"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Optional prereqs (won't hard-fail if missing)
for prereq in [
    PROJECT_ROOT / "artifacts" / "NB01" / "env_spec_apple.json",
    PROJECT_ROOT / "artifacts" / "NB02" / "obs_breakdown.json",
]:
    if prereq.exists():
        print(f"✅ Found prereq: {prereq.relative_to(PROJECT_ROOT)}")
    else:
        print(f"⚠️ Missing prereq (ok for NB03): {prereq.relative_to(PROJECT_ROOT)}")


## Configuration


In [ ]:
CFG = {
    "seed": 42,

    # Env
    "env_id": "UnitreeG1PlaceAppleInBowlFullBody-v1",
    "obs_mode": "state",
    "control_mode": "pd_joint_delta_pos",
    "reward_mode": "dense",              # also tested: "normalized_dense"
    "render_mode": "rgb_array",
    "render_backend": "cpu",

    # Eval
    "n_test_episodes": 20,
    "max_steps_per_episode": 300,

    # Visualization / logging
    "capture_episode_index": 0,          # which episode to capture frames from
    "capture_every_k_steps": 5,          # frame stride
}

SEED = int(CFG["seed"])
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("✅ Config loaded:", json.dumps(CFG, indent=2))


## Step 1 — Apple Reward Contract

The Apple env uses 4-stage dense reward with balance penalties.
Maximum normalized reward ≈ 10 per step.


In [ ]:
# ---------------------------------------------------------------------
# Step 1 — Ground-Truth Reward Contract (auto-aligned to apple_fullbody_env)
# ---------------------------------------------------------------------

def _get(mod, name, default=None):
    return getattr(mod, name, default)

# Extract exported constants (ground truth)
W_REACH = float(_get(env_mod, "W_REACH", 1.0))
W_GRASP = float(_get(env_mod, "W_GRASP", 2.0))
W_PLACE = float(_get(env_mod, "W_PLACE", 5.0))
W_TIME  = float(_get(env_mod, "W_TIME", 0.01))
W_JERK  = float(_get(env_mod, "W_JERK", 0.05))
W_ACT   = float(_get(env_mod, "W_ACT", 0.005))
SUCCESS_BONUS   = float(_get(env_mod, "SUCCESS_BONUS", 50.0))
SUCCESS_DIST    = float(_get(env_mod, "SUCCESS_DIST", 0.05))
GRASP_THRESHOLD = float(_get(env_mod, "GRASP_THRESHOLD", 0.01))

# Placing target is hard-coded in env reward (bowl_pos + [0,0,0.15])
GOAL_Z_OFFSET = 0.15

MAX_POSITIVE_REWARD = W_REACH + W_GRASP + W_PLACE + SUCCESS_BONUS

apple_reward_contract = {
    "version": "3.1-groundtruth",
    "env_id": CFG["env_id"],
    "robot": "unitree_g1_fullbody_fixed (37 DOF, fix_root_link=True, balance_passive_force=True)",
    "control_mode": CFG["control_mode"],
    "reward_modes": ["dense", "normalized_dense", "sparse", "none"],
    "max_positive_reward": float(MAX_POSITIVE_REWARD),  # 58.0 by default

    "stages": [
        {"name": "reaching",
         "formula": f"{W_REACH} * (1 - tanh(5 * ||tcp_pos - apple_pos||))",
         "max": W_REACH, "sign": "+"},

        {"name": "grasp",
         "formula": f"{W_GRASP} * 1[||tcp_pos-apple_pos|| < {GRASP_THRESHOLD}]",
         "max": W_GRASP, "sign": "+"},

        {"name": "place",
         "formula": f"{W_PLACE} * is_grasped * (1 - tanh(5 * ||apple_pos - (bowl_pos + [0,0,{GOAL_Z_OFFSET}])||))",
         "max": W_PLACE, "sign": "+"},

        {"name": "success_bonus",
         "formula": f"{SUCCESS_BONUS} if success else 0",
         "max": SUCCESS_BONUS, "sign": "+"},
    ],

    "penalties": [
        {"name": "time", "formula": f"-{W_TIME} per step", "sign": "-"},
        {"name": "jerk", "formula": f"-{W_JERK} * ||a_t - a_(t-1)||^2", "sign": "-"},
        {"name": "act",  "formula": f"-{W_ACT} * ||a_t||^2", "sign": "-"},
    ],

    "success_condition": f"dist(apple_pos, bowl_pos) <= {SUCCESS_DIST} and fail == False",
    "failure_conditions": [
        "fail == True (NaN / simulation explosion)",
        "timeout (max_episode_steps exceeded)",
    ],

    "normalized_dense_definition": f"normalized_dense = dense / {MAX_POSITIVE_REWARD}",
}

# Save contract
with open(ARTIFACTS_DIR / "reward_contract_apple_groundtruth.json", "w") as f:
    json.dump(apple_reward_contract, f, indent=2)

print("✅ Wrote reward contract:", ARTIFACTS_DIR / "reward_contract_apple_groundtruth.json")
print("max_positive_reward =", MAX_POSITIVE_REWARD)


## Step 2 — DishWipe Reward Contract (Reference for NB09)

The DishWipe env uses 9 terms + 2 balance terms. This is documented here
for reference — it will be used in NB09 (bonus task).


In [ ]:
dishwipe_reward_contract = {
    "version": "3.0",
    "env_id": "UnitreeG1DishWipeFullBody-v1",
    "robot": "unitree_g1 (37 DOF, full body, free root)",
    "terms": [
        {"name": "r_clean",   "weight": 10.0,  "sign": "+", "desc": "delta_clean progress"},
        {"name": "r_reach",   "weight": 0.5,   "sign": "+", "desc": "1-tanh(5*dist)"},
        {"name": "r_contact", "weight": 1.0,   "sign": "+", "desc": "is_contacting plate"},
        {"name": "r_sweep",   "weight": 0.3,   "sign": "+", "desc": "lateral movement"},
        {"name": "r_time",    "weight": 0.01,  "sign": "-", "desc": "per step cost"},
        {"name": "r_jerk",    "weight": 0.05,  "sign": "-", "desc": "jerk^2"},
        {"name": "r_act",     "weight": 0.005, "sign": "-", "desc": "action_norm^2"},
        {"name": "r_force",   "weight": 0.01,  "sign": "-", "desc": "excess force"},
        {"name": "r_success", "weight": 50.0,  "sign": "+", "desc": "one-shot at 95%"},
        {"name": "r_balance", "weight": "TBD", "sign": "-", "desc": "penalty for tilt"},
        {"name": "r_fall",    "weight": "TBD", "sign": "-", "desc": "terminate if fallen"},
    ],
    "note": "Used in NB09 only (bonus task)",
}

with open(ARTIFACTS_DIR / "reward_contract_dishwipe.json", "w") as f:
    json.dump(dishwipe_reward_contract, f, indent=2)

print("DishWipe Reward Contract (for NB09):")
for t in dishwipe_reward_contract["terms"]:
    print(f"  {t['sign']} {t['name']:12s}: w={str(t['weight']):>5s}  {t['desc']}")


## Step 3 — Validate Reward with Test Episodes

Run random actions for several episodes. Verify reward is dense
(most steps have non-zero reward) and bounded.


In [ ]:
# ---------------------------------------------------------------------
# Step 3 — Evaluation Rollouts + Reward Decomposition + Error Analysis
# ---------------------------------------------------------------------
import math
import imageio.v2 as imageio

def make_env(reward_mode: str):
    env = gym.make(
        CFG["env_id"],
        num_envs=1,
        obs_mode=CFG["obs_mode"],
        reward_mode=reward_mode,
        control_mode=CFG["control_mode"],
        render_mode=CFG["render_mode"],
        render_backend=CFG["render_backend"],
    )
    return CPUGymWrapper(env)

def _to_np(x):
    if x is None:
        return None
    if isinstance(x, np.ndarray):
        return x
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    try:
        return np.array(x)
    except Exception:
        return None

def deep_find_key(obj, key: str):
    """Search nested dicts/lists for a key and return the value (first match)."""
    if isinstance(obj, dict):
        if key in obj:
            return obj[key]
        for v in obj.values():
            r = deep_find_key(v, key)
            if r is not None:
                return r
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            r = deep_find_key(v, key)
            if r is not None:
                return r
    return None

def squeeze1(x):
    """Convert (1,D) -> (D,), keep scalars."""
    x = _to_np(x)
    if x is None:
        return None
    x = np.asarray(x)
    if x.ndim == 0:
        return float(x)
    if x.ndim >= 2 and x.shape[0] == 1:
        return np.asarray(x[0])
    return x

# ------------------------------
# Policies (action_space-aware)
# ------------------------------
def policy_random(obs, info, t, prev_action, action_space):
    return action_space.sample()

def make_policy_smooth_random(alpha=0.9):
    def _pi(obs, info, t, prev_action, action_space):
        a = action_space.sample()
        a = alpha * prev_action + (1.0 - alpha) * a
        return np.clip(a, action_space.low, action_space.high)
    return _pi

# ------------------------------
# Rollout runner
# ------------------------------
def rollout_eval(
    reward_mode: str,
    policy_name: str,
    policy_fn,
    n_episodes: int,
    max_steps: int,
    capture_episode_index: int = 0,
    capture_every_k_steps: int = 5,
):
    env = make_env(reward_mode)
    action_dim = int(np.prod(env.action_space.shape))

    all_info_keys = set()
    episodes = []
    captured_frames = []
    captured_trace = []

    for ep in range(n_episodes):
        obs, info = env.reset(seed=SEED + ep * 100)
        prev_action = np.zeros(action_dim, dtype=np.float32)

        ep_steps = []
        terminated = truncated = False
        term_reason = None

        for t in range(max_steps):
            a = policy_fn(obs, info, t, prev_action, env.action_space).astype(np.float32)

            obs2, rew, terminated, truncated, info2 = env.step(a)
            all_info_keys.update(info2.keys())

            # Extract extras from observation (robust)
            tcp_to_apple = squeeze1(deep_find_key(obs2, "tcp_to_apple"))
            apple_pos    = squeeze1(deep_find_key(obs2, "apple_pos"))
            bowl_pos     = squeeze1(deep_find_key(obs2, "bowl_pos"))
            is_grasped   = squeeze1(deep_find_key(obs2, "is_grasped"))

            # Convert to safe defaults
            dist_reach = float(np.linalg.norm(tcp_to_apple)) if tcp_to_apple is not None else float("nan")

            if apple_pos is not None and bowl_pos is not None:
                goal_pos = np.asarray(bowl_pos, dtype=np.float32) + np.array([0.0, 0.0, GOAL_Z_OFFSET], dtype=np.float32)
                dist_place = float(np.linalg.norm(np.asarray(apple_pos) - goal_pos))
            else:
                dist_place = float("nan")

            is_grasped_f = float(is_grasped) if is_grasped is not None else (1.0 if (not math.isnan(dist_reach) and dist_reach < GRASP_THRESHOLD) else 0.0)

            # Info flags (usually batched tensors)
            success = squeeze1(info2.get("success", False))
            fail    = squeeze1(info2.get("fail", False))
            success_b = bool(success) if isinstance(success, (bool, int, float)) else bool(np.asarray(success).item())
            fail_b    = bool(fail)    if isinstance(fail, (bool, int, float)) else bool(np.asarray(fail).item())

            # Reward decomposition (predicted)
            r_reach   = W_REACH * (1.0 - math.tanh(5.0 * dist_reach)) if not math.isnan(dist_reach) else float("nan")
            r_grasp   = W_GRASP * is_grasped_f
            r_place   = W_PLACE * is_grasped_f * (1.0 - math.tanh(5.0 * dist_place)) if not math.isnan(dist_place) else float("nan")
            r_success = SUCCESS_BONUS * (1.0 if success_b else 0.0)

            r_time = -W_TIME
            jerk = a - prev_action
            r_jerk = -W_JERK * float(np.sum(jerk ** 2))
            r_act  = -W_ACT  * float(np.sum(a ** 2))

            r_pred = r_reach + r_grasp + r_place + r_success + r_time + r_jerk + r_act

            step_row = {
                "t": t,
                "reward": float(rew),
                "r_pred": float(r_pred) if not (math.isnan(r_pred)) else float("nan"),
                "r_err": float(rew) - float(r_pred) if not (math.isnan(r_pred)) else float("nan"),
                "dist_reach": dist_reach,
                "dist_place": dist_place,
                "is_grasped": is_grasped_f,
                "success": success_b,
                "fail": fail_b,
                "action_norm": float(np.linalg.norm(a)),
                "jerk_norm": float(np.linalg.norm(jerk)),
            }
            ep_steps.append(step_row)

            # Capture frames for a single chosen episode
            if ep == capture_episode_index and (t % capture_every_k_steps == 0):
                try:
                    frame = env.render()
                    if frame is not None:
                        captured_frames.append(frame)
                        captured_trace.append(step_row)
                except Exception:
                    pass

            prev_action = a
            obs, info = obs2, info2

            if terminated or truncated:
                if terminated:
                    if success_b:
                        term_reason = "success"
                    elif fail_b:
                        term_reason = "fail_nan"
                    else:
                        term_reason = "terminated_other"
                else:
                    term_reason = "timeout"
                break

        ep_total = float(np.nansum([s["reward"] for s in ep_steps]))
        ep_mean = float(np.nanmean([s["reward"] for s in ep_steps]))
        ep_len = len(ep_steps)

        episodes.append({
            "episode": ep,
            "reward_mode": reward_mode,
            "policy": policy_name,
            "len": ep_len,
            "return": ep_total,
            "mean_reward": ep_mean,
            "termination": term_reason or "max_steps",
            "success": (term_reason == "success"),
            "fail_nan": (term_reason == "fail_nan"),
            "timeout": (term_reason == "timeout"),
            "max_action_norm": float(np.nanmax([s["action_norm"] for s in ep_steps])) if ep_steps else 0.0,
            "max_jerk_norm": float(np.nanmax([s["jerk_norm"] for s in ep_steps])) if ep_steps else 0.0,
            "trace": ep_steps,
        })

    env.close()
    return {
        "reward_mode": reward_mode,
        "policy": policy_name,
        "episodes": episodes,
        "all_info_keys": sorted(list(all_info_keys)),
        "captured_frames": captured_frames,
        "captured_trace": captured_trace,
    }

# ------------------------------
# Run evaluations
# ------------------------------
smooth_pi = make_policy_smooth_random(alpha=0.9)

# Dense reward (ground truth scale)
eval_dense_random = rollout_eval(
    reward_mode="dense",
    policy_name="random",
    policy_fn=policy_random,
    n_episodes=CFG["n_test_episodes"],
    max_steps=CFG["max_steps_per_episode"],
    capture_episode_index=CFG["capture_episode_index"],
    capture_every_k_steps=CFG["capture_every_k_steps"],
)

eval_dense_smooth = rollout_eval(
    reward_mode="dense",
    policy_name="smooth_random",
    policy_fn=smooth_pi,
    n_episodes=CFG["n_test_episodes"],
    max_steps=CFG["max_steps_per_episode"],
    capture_episode_index=CFG["capture_episode_index"],
    capture_every_k_steps=CFG["capture_every_k_steps"],
)

# Normalized dense sanity check (should be ~dense / 58, plus penalties)
eval_norm_smooth = rollout_eval(
    reward_mode="normalized_dense",
    policy_name="smooth_random",
    policy_fn=smooth_pi,
    n_episodes=max(5, CFG["n_test_episodes"] // 4),
    max_steps=CFG["max_steps_per_episode"],
    capture_episode_index=CFG["capture_episode_index"],
    capture_every_k_steps=CFG["capture_every_k_steps"],
)

# Persist raw eval dicts
with open(ARTIFACTS_DIR / "eval_dense_random.json", "w") as f:
    json.dump(eval_dense_random, f, indent=2)
with open(ARTIFACTS_DIR / "eval_dense_smooth.json", "w") as f:
    json.dump(eval_dense_smooth, f, indent=2)
with open(ARTIFACTS_DIR / "eval_norm_smooth.json", "w") as f:
    json.dump(eval_norm_smooth, f, indent=2)

print("✅ Saved eval JSONs to", ARTIFACTS_DIR)

# Quick summary printer
def summarize_eval(ev):
    eps = ev["episodes"]
    summary = {
        "policy": ev["policy"],
        "reward_mode": ev["reward_mode"],
        "n_eps": len(eps),
        "success_rate": float(np.mean([e["success"] for e in eps])),
        "fail_nan_rate": float(np.mean([e["fail_nan"] for e in eps])),
        "timeout_rate": float(np.mean([e["timeout"] for e in eps])),
        "mean_return": float(np.mean([e["return"] for e in eps])),
        "mean_len": float(np.mean([e["len"] for e in eps])),
    }
    return summary

summary_dense_random = summarize_eval(eval_dense_random)
summary_dense_smooth = summarize_eval(eval_dense_smooth)
summary_norm_smooth  = summarize_eval(eval_norm_smooth)

summary_all = {
    "dense_random": summary_dense_random,
    "dense_smooth_random": summary_dense_smooth,
    "normalized_dense_smooth_random": summary_norm_smooth,
}

print("=== SUMMARY ===")
print(json.dumps(summary_all, indent=2))

with open(ARTIFACTS_DIR / "summary.json", "w") as f:
    json.dump(summary_all, f, indent=2)

# Expose for later cells
all_info_keys = set(eval_dense_smooth["all_info_keys"])
termination_counts = {
    "success": int(sum(e["success"] for e in eval_dense_smooth["episodes"])),
    "fail_nan": int(sum(e["fail_nan"] for e in eval_dense_smooth["episodes"])),
    "timeout": int(sum(e["timeout"] for e in eval_dense_smooth["episodes"])),
    "other": int(sum(e["termination"] not in ["success","fail_nan","timeout"] for e in eval_dense_smooth["episodes"])),
}


## Step 4 — Info Dict Keys


In [ ]:
# ---------------------------------------------------------------------
# Step 4 — Inspect info keys (ground truth interface)
# ---------------------------------------------------------------------
info_data = {"available_keys": sorted(list(all_info_keys))}
print("Info keys (from eval):")
for k in info_data["available_keys"]:
    print(" -", k)

with open(ARTIFACTS_DIR / "info_keys.json", "w") as f:
    json.dump(info_data, f, indent=2)
print("✅ Wrote", ARTIFACTS_DIR / "info_keys.json")


## Step 5 — Safety Validation (Fall Detection)

Verify that `is_fallen()` triggers episode termination with random actions.


In [ ]:
# ---------------------------------------------------------------------
# Step 5 — Safety validation (NaN / explosion vs timeout)
# ---------------------------------------------------------------------
# In this env, the ground-truth failure signal is `info["fail"]` (NaN states).
# "Fall detection" is NOT the primary failure mode here because the root is fixed.

safety_results = {
    "primary_failure_mode": "fail_nan (info['fail'] from evaluate())",
    "termination_counts_from_dense_smooth": termination_counts,
    "notes": [
        "If fail_nan rate is high, reduce exploration magnitude, add action smoothing, or tighten physics stability checks.",
        "For training: consider reward_mode='normalized_dense' to keep return scale stable across algorithms.",
    ],
}

print("Safety Validation:")
print(json.dumps(safety_results, indent=2))

with open(ARTIFACTS_DIR / "safety_validation.json", "w") as f:
    json.dump(safety_results, f, indent=2)
print("✅ Wrote", ARTIFACTS_DIR / "safety_validation.json")


## Step 6 — MLflow Helper Utilities

Define reusable helper functions for NB04–NB09.


In [ ]:
# ── MLflow Helpers (reusable pattern) ──

def setup_mlflow(experiment_name="g1_fullbody_apple_dishwipe"):
    """Set up MLflow tracking. Returns True if successful."""
    try:
        import mlflow
        from dotenv import load_dotenv
        load_dotenv(Path.cwd() / ".env.local")
        uri = os.environ.get("MLFLOW_TRACKING_URI", "")
        if uri:
            mlflow.set_tracking_uri(uri)
            mlflow.set_experiment(experiment_name)
            return True
    except Exception:
        pass
    return False

def log_training_run(run_name, params, metrics=None, artifacts_dir=None):
    """Log a training run to MLflow."""
    try:
        import mlflow
        with mlflow.start_run(run_name=run_name):
            mlflow.log_params(params)
            if metrics:
                mlflow.log_metrics(metrics)
            if artifacts_dir:
                mlflow.log_artifacts(str(artifacts_dir))
    except Exception as e:
        print(f"MLflow logging failed: {e}")

def log_eval_run(run_name, eval_dict, artifacts_dir=None):
    """Log an evaluation run to MLflow."""
    try:
        import mlflow
        with mlflow.start_run(run_name=run_name):
            mlflow.log_dict(eval_dict, "eval_results.json")
            if artifacts_dir:
                mlflow.log_artifacts(str(artifacts_dir))
    except Exception as e:
        print(f"MLflow logging failed: {e}")

print("✅ MLflow helper functions defined")
print("  - setup_mlflow(experiment_name)")
print("  - log_training_run(run_name, params, metrics, artifacts_dir)")
print("  - log_eval_run(run_name, eval_dict, artifacts_dir)")


## Plots


In [ ]:
# ---------------------------------------------------------------------
# Visualizations (pretty + informative)
# ---------------------------------------------------------------------

def flatten_steps(ev):
    steps = []
    for e in ev["episodes"]:
        steps.extend(e["trace"])
    return steps

steps_dense_smooth = flatten_steps(eval_dense_smooth)
rewards = np.array([s["reward"] for s in steps_dense_smooth], dtype=np.float32)
r_err   = np.array([s["r_err"] for s in steps_dense_smooth], dtype=np.float32)
a_norm  = np.array([s["action_norm"] for s in steps_dense_smooth], dtype=np.float32)
j_norm  = np.array([s["jerk_norm"] for s in steps_dense_smooth], dtype=np.float32)

# Figure 1 — Summary
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].hist(rewards[np.isfinite(rewards)], bins=60)
axes[0, 0].set_title("Reward distribution (dense, smooth_random)")
axes[0, 0].set_xlabel("reward")

axes[0, 1].hist(r_err[np.isfinite(r_err)], bins=60)
axes[0, 1].set_title("Reward decomposition error (env - predicted)")
axes[0, 1].set_xlabel("reward_error")

axes[1, 0].bar(list(termination_counts.keys()), list(termination_counts.values()))
axes[1, 0].set_title("Termination breakdown (dense, smooth_random)")

# True stage maxima
stages = ["reach", "grasp", "place", "success_bonus"]
max_rewards = [W_REACH, W_GRASP, W_PLACE, SUCCESS_BONUS]
axes[1, 1].bar(stages, max_rewards)
axes[1, 1].set_title("Ground-truth stage maxima (+)")
axes[1, 1].set_ylabel("max reward per stage")

fig.suptitle("NB03 — Reward & Safety (Ground Truth)", fontsize=14)
fig.tight_layout()
fig_path = ARTIFACTS_DIR / "nb03_summary_plots.png"
fig.savefig(fig_path, dpi=160)
plt.show()
print("✅ Saved", fig_path)

# Figure 2 — Captured episode time-series
cap_trace = eval_dense_smooth["captured_trace"]
if len(cap_trace) > 0:
    t = np.array([s["t"] for s in cap_trace])
    r = np.array([s["reward"] for s in cap_trace], dtype=np.float32)
    rp = np.array([s["r_pred"] for s in cap_trace], dtype=np.float32)
    dr = np.array([s["dist_reach"] for s in cap_trace], dtype=np.float32)
    dp = np.array([s["dist_place"] for s in cap_trace], dtype=np.float32)
    an = np.array([s["action_norm"] for s in cap_trace], dtype=np.float32)
    jn = np.array([s["jerk_norm"] for s in cap_trace], dtype=np.float32)

    fig2, ax = plt.subplots(figsize=(14, 4))
    ax.plot(t, r, label="env reward")
    ax.plot(t, rp, label="pred reward", alpha=0.8)
    ax.set_title("Captured episode — reward vs predicted")
    ax.set_xlabel("t")
    ax.legend()
    p = ARTIFACTS_DIR / "episode_reward_vs_pred.png"
    fig2.tight_layout()
    fig2.savefig(p, dpi=160)
    plt.show()
    print("✅ Saved", p)

    fig3, ax = plt.subplots(figsize=(14, 4))
    ax.plot(t, dr, label="dist(tcp, apple)")
    ax.plot(t, dp, label="dist(apple, goal)")
    ax.set_title("Captured episode — distances")
    ax.set_xlabel("t")
    ax.legend()
    p = ARTIFACTS_DIR / "episode_distances.png"
    fig3.tight_layout()
    fig3.savefig(p, dpi=160)
    plt.show()
    print("✅ Saved", p)

    fig4, ax = plt.subplots(figsize=(14, 4))
    ax.plot(t, an, label="||a||")
    ax.plot(t, jn, label="||jerk||")
    ax.set_title("Captured episode — action/jerk norms")
    ax.set_xlabel("t")
    ax.legend()
    p = ARTIFACTS_DIR / "episode_action_jerk.png"
    fig4.tight_layout()
    fig4.savefig(p, dpi=160)
    plt.show()
    print("✅ Saved", p)

# Filmstrip + GIF
frames = eval_dense_smooth["captured_frames"]
if len(frames) > 0:
    # Filmstrip (12 frames evenly spaced)
    n = len(frames)
    k = 12 if n >= 12 else n
    idxs = np.linspace(0, n - 1, k).astype(int).tolist()

    cols = 4
    rows = int(np.ceil(k / cols))
    fig5, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for ax_i, fi in enumerate(idxs):
        ax = axes[ax_i]
        img = frames[fi]
        ax.imshow(img)
        if fi < len(cap_trace):
            s = cap_trace[fi]
            txt = f"t={s['t']} r={s['reward']:.2f}\nreach={s['dist_reach']:.3f} place={s['dist_place']:.3f}\n||a||={s['action_norm']:.2f} ||j||={s['jerk_norm']:.2f}"
            ax.text(5, 15, txt, fontsize=9, color="white",
                    bbox=dict(facecolor="black", alpha=0.5, pad=3))
        ax.axis("off")

    for j in range(len(idxs), len(axes)):
        axes[j].axis("off")

    fp = ARTIFACTS_DIR / "episode_filmstrip.png"
    fig5.tight_layout()
    fig5.savefig(fp, dpi=160)
    plt.show()
    print("✅ Saved", fp)

    # GIF
    try:
        gif_path = ARTIFACTS_DIR / "episode.gif"
        imageio.mimsave(gif_path, frames, fps=max(1, 30 // int(CFG["capture_every_k_steps"])))
        print("✅ Saved", gif_path)
    except Exception as e:
        print("⚠️ GIF save failed:", e)


## Error Analysis (SOTA-grade)

This section turns rollouts into a *debuggable* report:
- Outlier steps where reward decomposition mismatch is large
- Episodes that ended with `fail_nan` (simulation explosion)
- Action/jerk statistics to diagnose instability


In [ ]:
import csv

# Use smooth-random dense eval as the main diagnostic target
eps = eval_dense_smooth["episodes"]

# 1) Step-level outliers (reward decomposition mismatch)
all_steps = []
for e in eps:
    for s in e["trace"]:
        row = dict(s)
        row["episode"] = e["episode"]
        row["termination"] = e["termination"]
        all_steps.append(row)

err_abs = np.array([abs(r["r_err"]) for r in all_steps if np.isfinite(r["r_err"])], dtype=np.float32)
err_abs_mean = float(np.mean(err_abs)) if err_abs.size else float("nan")
err_abs_p95  = float(np.quantile(err_abs, 0.95)) if err_abs.size else float("nan")
err_abs_p99  = float(np.quantile(err_abs, 0.99)) if err_abs.size else float("nan")

# pick top 30 by abs error
top_steps = sorted(
    [r for r in all_steps if np.isfinite(r["r_err"])],
    key=lambda x: abs(x["r_err"]),
    reverse=True,
)[:30]

# Save outlier steps to CSV
csv_path = ARTIFACTS_DIR / "outlier_steps_top30.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=[
        "episode","t","reward","r_pred","r_err",
        "dist_reach","dist_place","is_grasped","success","fail",
        "action_norm","jerk_norm","termination",
    ])
    w.writeheader()
    for r in top_steps:
        w.writerow({k: r.get(k) for k in w.fieldnames})
print("✅ Saved", csv_path)

# 2) Episode-level failure diagnostics
fail_eps = []
for e in eps:
    if e["fail_nan"]:
        trace = e["trace"]
        # first step where fail==True OR reward == -1.0
        fail_t = None
        for s in trace:
            if s.get("fail", False) or (isinstance(s.get("reward", None), (int,float)) and float(s["reward"]) <= -0.999):
                fail_t = int(s["t"])
                break
        fail_eps.append({
            "episode": e["episode"],
            "fail_t": fail_t,
            "len": e["len"],
            "return": e["return"],
            "max_action_norm": e["max_action_norm"],
            "max_jerk_norm": e["max_jerk_norm"],
        })

# 3) Normalized reward sanity range (should be around dense/58 with penalties)
norm_steps = []
for e in eval_norm_smooth["episodes"]:
    norm_steps.extend(e["trace"])
norm_rewards = np.array([s["reward"] for s in norm_steps], dtype=np.float32)
norm_summary = {
    "min": float(np.nanmin(norm_rewards)) if norm_rewards.size else None,
    "max": float(np.nanmax(norm_rewards)) if norm_rewards.size else None,
    "mean": float(np.nanmean(norm_rewards)) if norm_rewards.size else None,
}

error_report = {
    "reward_decomposition_error": {
        "mean_abs": err_abs_mean,
        "p95_abs": err_abs_p95,
        "p99_abs": err_abs_p99,
        "note": "If these are large, observation extraction may be missing keys OR env reward definition changed.",
    },
    "fail_nan_episodes": {
        "count": len(fail_eps),
        "examples": fail_eps[:10],
        "note": "fail_nan indicates NaN states / simulation explosion (ground truth from env.evaluate()).",
    },
    "action_jerk_stats_dense_smooth": {
        "action_norm_mean": float(np.mean(a_norm[np.isfinite(a_norm)])) if a_norm.size else None,
        "action_norm_p95": float(np.quantile(a_norm[np.isfinite(a_norm)], 0.95)) if a_norm.size else None,
        "jerk_norm_mean": float(np.mean(j_norm[np.isfinite(j_norm)])) if j_norm.size else None,
        "jerk_norm_p95": float(np.quantile(j_norm[np.isfinite(j_norm)], 0.95)) if j_norm.size else None,
    },
    "normalized_dense_reward_range": norm_summary,
}

with open(ARTIFACTS_DIR / "error_analysis.json", "w") as f:
    json.dump(error_report, f, indent=2)

print("=== ERROR ANALYSIS REPORT ===")
print(json.dumps(error_report, indent=2))
print("✅ Wrote", ARTIFACTS_DIR / "error_analysis.json")


## Save Artifacts


In [ ]:
# ---------------------------------------------------------------------
# Save Artifacts + (Optional) log to MLflow
# ---------------------------------------------------------------------
with open(ARTIFACTS_DIR / "nb03_config.json", "w") as f:
    json.dump(CFG, f, indent=2)

# Close the temporary env created for policy closures (if still alive)
try:
    env.close()
except Exception:
    pass

# Optional MLflow logging
mlflow_ok = setup_mlflow(experiment_name="g1_fullbody_apple_reward_safety")
print("MLflow enabled:", mlflow_ok)

if mlflow_ok:
    eval_payload = {
        "summary": summary_all,
        "termination_counts_dense_smooth": termination_counts,
        "normalized_dense_reward_range": error_report.get("normalized_dense_reward_range", {}),
    }
    # Log a compact eval dict + full artifacts folder
    log_eval_run(
        run_name=f"NB03_eval_seed{SEED}",
        eval_dict=eval_payload,
        artifacts_dir=ARTIFACTS_DIR,
    )

print("✅ NB03 complete. Artifacts saved to:", ARTIFACTS_DIR)


## Artifacts

| File | Description |
|------|-------------|
| `reward_contract_apple.json` | Apple 4-stage reward formal contract |
| `reward_contract_dishwipe.json` | DishWipe 9+2 term reward contract (NB09 ref) |
| `reward_validation.json` | Reward range/distribution stats |
| `safety_validation.json` | Fall detection test results |
| `info_keys.json` | Available keys in env info dict |
| `reward_analysis.png` | Reward distribution + termination plots |
